In [102]:
from bs4 import BeautifulSoup
import requests
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
import json
from IPython.display import Markdown, display
openai = OpenAI()

from scraper import fetch_website_links

In [31]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

In [141]:


def parse_url_links(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    section = soup.find("main")
    # ("div", {"data-parsely-slot": "Sports-Top Headlines"})

    links = [
        {
            "url": link.get("href"),
            "title": link.get_text(strip=True)
        }
        for link in section.find_all("a")
        if link.get("href") and link.get_text(strip=True)
    ]
           
    return json.dumps(links)

In [142]:
print(parse_url_links("https://www.apnews.com/sports/"))


[{"url": "https://apnews.com/article/masters-scottie-scheffler-4bec0577797efd4563047c57381d0428", "title": "Golf has been secondary for Scottie Scheffler of late. It\u2019s hard to know what to expect at Masters"}, {"url": "https://apnews.com/article/masters-scottie-scheffler-4bec0577797efd4563047c57381d0428", "title": "Scottie Scheffler has two kids and two Masters titles. He\u2019s the favorite in this week\u2019s tournament, as he tends to be for all majors these days."}, {"url": "https://apnews.com/article/pittsburgh-pirates-konnor-griffin-e31a7c4d4b8a5374c23e79d65926770c", "title": "Pirates sign teenage shortstop Konnor Griffin to a 9-year deal worth at least $140 million"}, {"url": "https://apnews.com/article/falcons-james-pearce-nfl-974a71ca1fb7fba8dc44c5cdc1d6df9b", "title": "James Pearce Jr. not at Falcons\u2019 voluntary offseason workouts, coach Kevin Stefanski says"}, {"url": "https://apnews.com/article/free-agency-wnba-cba-625b65d3a47ea2e7e721a0d1911097fa", "title": "WNBA 

In [146]:
system_prompt = """
You are an experineced journalist working in the sports department, you are very funny and 
can make hilarious jokes about specific news you are writing about.



"""
def parse_links_user_prompt(url):
    user_prompt = f"""
  Here is a list of links and their title on the website {url}, 
  I want you to play the role of a journalist and write funny,not so long article,
  The article will need to include all the the titles, but not the links, 
  

  """
    titles = parse_url_links(url)
    user_prompt += "\n".join(titles.split(","))
    return user_prompt
    


    

 
  

In [147]:
prmpt = parse_links_user_prompt("https://www.apnews.com/sports/")
print(prmpt)


  Here is a list of links and their title on the website https://www.apnews.com/sports/, 
  I want you to play the role of a journalist and write funny,not so long article,
  The article will need to include all the the titles, but not the links, 


  [{"url": "https://apnews.com/article/soler-lopez-fight-suspended-44fd5165f1cfab921499149dc0b51262"
 "title": "Angels\u2019 Jorge Soler and Braves\u2019 Reynaldo L\u00f3pez receive suspensions following brawl"}
 {"url": "https://apnews.com/article/soler-lopez-fight-suspended-44fd5165f1cfab921499149dc0b51262"
 "title": "Los Angeles Angels designated hitter Jorge Soler and Atlanta Braves pitcher Reynaldo L\u00f3pez each received seven-game suspensions from Major League Baseball on Wednesday after they were ejected following their participation in a brawl."}
 {"url": "https://apnews.com/article/masters-scottie-scheffler-4bec0577797efd4563047c57381d0428"
 "title": "Golf has been secondary for Scottie Scheffler of late. It\u2019s hard to know 

In [148]:
def the_article(url):
  response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": parse_links_user_prompt(url)}
  ]  
  )
  result = response.choices[0].message.content
  return result


In [149]:
final = the_article("https://www.apnews.com/sports/")
display(Markdown(final))

**In Sports Today: A Brawl, A Child Prodigy, and Some Baffling Decisions!**

Have you ever noticed that sports can feel like a soap opera? Well, buckle up, because things are about to get dramatic, hilarious, and a little confusing!

First up, we have the fiery saga of *Angels’ Jorge Soler and Braves’ Reynaldo López receive suspensions following brawl*. Yes, folks, these two gentlemen were ejected after their boxing match masquerading as a baseball game, leading Major League Baseball to hand out some serious suspensions—seven games each. Let’s just say, the only “angels” here were the ones praying the brawl wouldn’t leave them grounded!

In the golf world, *Golf has been secondary for Scottie Scheffler of late*. I mean, who needs to play golf when you can contemplate deep philosophical questions like, “Why do I even need to hit a tiny ball into a hole when I can just yell ‘fore’ and get noticed?”

Meanwhile, *8-year-old Frankie Fleetwood steals the show during Par 3 Contest on the eve of the Masters*. Seriously, a child stole the spotlight from grown adults swinging clubs! That’s right, little Frankie could probably negotiate a multi-million dollar deal at this point—who needs agents?

Speaking of negotiations, we have *James Pearce Jr. not at Falcons’ voluntary offseason workouts*. Sounds like an interesting interpretation of “voluntary,” which is basically code for “I decided to hit snooze instead.” 

In another corner of the sports arena, *WNBA free agency opens with $1.4 million franchise tags for Ionescu, Collier, and Plum*. That’s right—who needs “Teacher of the Year” awards when you can slap a $1.4 million price on your best athletes? Forget about dreams; this is a post-midnight fantasy!

Switching gears to the transfer portal saga, *Michigan gets its moment, then the transfer portal opens and the scramble for 2027 begins*. It’s like one big game of musical chairs, but instead of chairs, they’re all vying for scholarships and snacks from the vending machine.

Now, for a bit more serious news, *Colorado QB’s blood alcohol level was twice legal limit in fatal single-car crash report reveals*. Let’s just hope he’s learning to steer clear of trouble while keeping his car on the road and himself out of the headlines for the wrong reasons.

In a tale of young talent, *Pirates sign teenage shortstop Konnor Griffin to a 9-year deal worth at least $140 million*. That’s right, the kid can finally replace the worn-out bed he’s been sleeping in since he was five. At this rate, he’ll probably need a financial advisor just as soon as he learns to spell “500 dollars.”

And can we talk about celebrity appearances? *Kevin Hart and Jason Kelce are among the celebrity caddies at Augusta National’s Par 3 Contest*. Because nothing says “serious golf” like a comedian handing you your putter. Here’s a suggestion: maybe they should have the celebs play the game, and whoever is the funniest gets to wear the green jacket!

At the end of the day, sports isn’t just about winning—it's about the spectacle, the drama, and the moments that make us laugh, cry, and shake our heads in wonder. From brawls on the field to kids stealing our hearts, it’s a wild ride. So, grab your popcorn and settle in; we’re just getting started!